# 02 — Silver: Route-Day Profitability

Bronze → **Silver**: dedupe on the natural key, add derived profitability &
efficiency metrics, and define **underperformance** from the *actual* margin
distribution (not an assumed number). Output:
`delta/silver/route_day_profitability`, partitioned by `year`, `region`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))

from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

from spark_session import build_spark, delta_path

spark = build_spark("02_silver_transform")
BRONZE = delta_path("bronze", "commercial_routes", spark)
SILVER = delta_path("silver", "route_day_profitability", spark)
print("Spark", spark.version)
print("BRONZE:", BRONZE)
print("SILVER:", SILVER)

:: loading settings :: url = jar:file:/Users/yashasvipamu/Documents/Python%20Files/temp_project/.venv/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/yashasvipamu/.ivy2.5.2/cache
The jars for the packages stored in: /Users/yashasvipamu/.ivy2.5.2/jars
io.delta#delta-spark_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-bb05b980-bf94-4861-a261-8e69f69f627c;1.0
	confs: [default]
	found io.delta#delta-spark_2.13;4.0.1 in central
	found io.delta#delta-storage;4.0.1 in central
	found org.antlr#antlr4-runtime;4.13.1 in central
:: resolution report :: resolve 117ms :: artifacts dl 4ms
	:: modules in use:
	io.delta#delta-spark_2.13;4.0.1 from central in [default]
	io.delta#delta-storage;4.0.1 from central in [default]
	org.antlr#antlr4-runtime;4.13.1 from central in [default]
	---------------------------------------------------------------------
	|                  

26/06/23 20:41:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark 4.0.3
BRONZE: /Users/yashasvipamu/Documents/Python Files/temp_project/gfl-route-profitability/delta/bronze/commercial_routes
SILVER: /Users/yashasvipamu/Documents/Python Files/temp_project/gfl-route-profitability/delta/silver/route_day_profitability


## Read Bronze + dedupe on `route_date_key`

The grain is one row per route-day. Bronze upserts restated keys, but a Silver
build should still be defensive: if the same `route_date_key` ever appears more
than once (e.g. a mid-load restatement), keep the **latest** by `ingestion_ts`.
This makes Silver deterministic and idempotent regardless of Bronze history.

In [2]:
bronze = spark.read.format("delta").load(BRONZE)

# Keep latest ingested row per natural key.
w = Window.partitionBy("route_date_key").orderBy(F.col("ingestion_ts").desc())
dedup = (
    bronze
    .withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

print("bronze rows           :", bronze.count())
print("after dedupe rows     :", dedup.count())
print("distinct keys (check) :", dedup.select("route_date_key").distinct().count())

# Normalize the raw incident sentinel: "None" string -> real null (see 01 DQ).
dedup = dedup.withColumn(
    "incident_type",
    F.when(F.col("incident_type") == "None", None).otherwise(F.col("incident_type")),
)
print("incident_type non-null after normalize:", dedup.filter(F.col("incident_type").isNotNull()).count())

26/06/23 20:41:16 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


bronze rows           : 12000


after dedupe rows     : 12000


distinct keys (check) : 12000


incident_type non-null after normalize: 297


## Part A — Derived metrics

### Why these metrics for a high-density commercial waste route (not generic BI)

A front-load commercial route is a **fixed-cost asset driven down a road to make
discrete billable stops**. Profit is destroyed in three specific ways, and each
derived metric targets one:

**1. Per-stop economics** — the route's revenue is the *sum of stops*, and the
truck/crew cost is largely fixed for the day. So the real question is whether each
completed stop pays its share.
- `revenue_/cost_/profit_per_completed_stop` — divide by **completed** (not total)
  stops: a missed stop earns nothing but still consumed drive time. A route with
  high `total_revenue` but low profit-per-stop is over-served (too many cheap stops
  for the labour burned).

**2. Density / drive efficiency** — front-load profit lives and dies on **route
density** (stops packed into few km). A sparse route burns fuel and hours between
bins.
- `revenue_/cost_/profit_per_km` and `fuel_litres_per_100km` — expose low-density
  routes whose distance cost isn't covered by billing.
- `stops_per_labour_hour` — the single best productivity proxy for a crew: a dense
  urban route does 15+ stops/hr; a thin industrial run does far fewer for the same
  wage.

**3. Disposal drag** — for waste specifically, **tipping fees scale with tonnage**,
not revenue. A route can bill well yet bleed margin if it hauls heavy, low-value
material (e.g. wet organics / general waste) to an expensive tip.
- `cost_/profit_/revenue_per_tonne` and especially `disposal_cost_per_tonne` —
  isolate routes where the *material economics* (gate fee vs. what the customer
  pays for that weight) are upside-down. This is the metric a generic BI dashboard
  misses.

**Operational health (leading indicators of the above):**
- `completion_rate`, `missed_stop_rate` — missed stops mean re-service trips
  (double fuel/labour, zero extra revenue) and SLA risk.
- **Cost-component shares** (`disposal/fuel/labour/maintenance/admin _cost_share`)
  — decompose *where* a thin-margin day's money goes, so Part 2's "primary cost
  driver" analysis is a direct lookup, not a re-derivation.

In [3]:
def safe_div(num, den):
    """Divide two columns, returning null when the denominator is null or 0
    (instead of throwing / producing inf). Keeps Silver free of divide-by-zero
    poison for routes with, e.g., 0 tonnes collected on a light day."""
    den_c = F.col(den) if isinstance(den, str) else den
    num_c = F.col(num) if isinstance(num, str) else num
    return F.when((den_c.isNull()) | (den_c == 0), None).otherwise(num_c / den_c)


silver = (
    dedup
    # 1. Per-stop economics (completed stops = the billable denominator)
    .withColumn("revenue_per_completed_stop", safe_div("total_revenue", "completed_stops"))
    .withColumn("cost_per_completed_stop",    safe_div("total_cost", "completed_stops"))
    .withColumn("profit_per_completed_stop",  safe_div("gross_profit", "completed_stops"))
    # 2. Density / drive efficiency
    .withColumn("revenue_per_km", safe_div("total_revenue", "total_distance_km"))
    .withColumn("cost_per_km",    safe_div("total_cost", "total_distance_km"))
    .withColumn("profit_per_km",  safe_div("gross_profit", "total_distance_km"))
    .withColumn("fuel_litres_per_100km",
                safe_div(F.col("total_fuel_used_litres") * 100, "total_distance_km"))
    .withColumn("stops_per_labour_hour", safe_div("completed_stops", "total_labour_hours"))
    # 3. Material / disposal economics
    .withColumn("revenue_per_tonne", safe_div("total_revenue", "total_tonnes"))
    .withColumn("cost_per_tonne",    safe_div("total_cost", "total_tonnes"))
    .withColumn("profit_per_tonne",  safe_div("gross_profit", "total_tonnes"))
    .withColumn("disposal_cost_per_tonne", safe_div("disposal_cost", "total_tonnes"))
    # Operational health
    .withColumn("completion_rate",  safe_div("completed_stops", "total_stops"))
    .withColumn("missed_stop_rate", safe_div("missed_stops", "total_stops"))
    # Cost-component shares of total_cost
    .withColumn("disposal_cost_share",    safe_div("disposal_cost", "total_cost"))
    .withColumn("fuel_cost_share",        safe_div("fuel_cost", "total_cost"))
    .withColumn("labour_cost_share",      safe_div("labour_cost", "total_cost"))
    .withColumn("maintenance_cost_share", safe_div("maintenance_cost", "total_cost"))
    .withColumn("admin_cost_share",       safe_div("admin_cost", "total_cost"))
)

print("Sample derived metrics (5 rows):")
silver.select(
    "route_date_key", "gross_margin_pct",
    "profit_per_completed_stop", "disposal_cost_per_tonne",
    "stops_per_labour_hour", "completion_rate", "disposal_cost_share",
).show(5, truncate=False)

Sample derived metrics (5 rows):


+--------------+----------------+-------------------------+-----------------------+---------------------+------------------+-------------------+
|route_date_key|gross_margin_pct|profit_per_completed_stop|disposal_cost_per_tonne|stops_per_labour_hour|completion_rate   |disposal_cost_share|
+--------------+----------------+-------------------------+-----------------------+---------------------+------------------+-------------------+
|RDK-0000001   |73.97           |40.24566176470588        |37.93062528525787      |9.024552090245521    |0.9855072463768116|0.43156030762679737|
|RDK-0000002   |-10.71          |-4.08781512605042        |67.29215446444174      |8.056872037914692    |1.0               |0.7655561077427692 |
|RDK-0000003   |32.7            |19.01563492063492        |82.77320291173794      |7.167235494880547    |0.984375          |0.7377353632303179 |
|RDK-0000004   |68.14           |34.07793103448276        |28.17467387846007      |8.070500927643785    |0.9720670391061452|0.3194

## Part B — Defining underperformance (reasoned, from real numbers)

### Step 1 — the actual `gross_margin_pct` distribution

In [4]:
qs = silver.approxQuantile("gross_margin_pct", [0.10, 0.25, 0.50, 0.75, 0.90], 0.001)
p10, p25, p50, p75, p90 = [round(x, 2) for x in qs]
stats = silver.select(
    F.round(F.mean("gross_margin_pct"), 2).alias("mean"),
    F.round(F.min("gross_margin_pct"), 2).alias("min"),
    F.round(F.max("gross_margin_pct"), 2).alias("max"),
).first()
n_rows = silver.count()
n_neg  = silver.filter(F.col("gross_profit") < 0).count()

print("=== gross_margin_pct distribution (all route-days) ===")
print(f"P10 : {p10}")
print(f"P25 : {p25}")
print(f"median (P50): {p50}")
print(f"P75 : {p75}")
print(f"P90 : {p90}")
print(f"mean: {stats['mean']}   min: {stats['min']}   max: {stats['max']}")
print(f"rows with negative gross_profit: {n_neg} ({round(100*n_neg/n_rows, 2)}%)")

=== gross_margin_pct distribution (all route-days) ===
P10 : 9.64
P25 : 28.51
median (P50): 46.99
P75 : 67.35
P90 : 76.14
mean: 44.75   min: -122.15   max: 90.35
rows with negative gross_profit: 717 (5.97%)


### Step 2 — chosen threshold and reasoning

**Threshold: a route-day is underperforming when `gross_margin_pct < 10%`.**

Reasoning, grounded in the percentiles printed directly above (P10 ≈ 9.6,
P25 ≈ 28.5, median ≈ 47, mean ≈ 44.8, ~6% negative):

- **It tracks the natural break in the data, not a round-number guess.** P10 sits
  at ~9.6%, so a 10% cut isolates almost exactly the **worst decile** of route-days
  — the genuine tail — while leaving the healthy middle (median ~47%) untouched. A
  threshold up near P25 (28%) would brand a quarter of perfectly normal routes as
  "failing" and drown the signal.
- **It fully contains the bleed.** Every negative-gross-profit day (~6%) has margin
  < 0 < 10%, so the flag captures all outright loss-makers **plus** the thin
  0–10% band that barely clears variable cost. On a commercial route a single fuel-
  price tick or one reweighed heavy load can flip a sub-10% day negative — so <10%
  is the practical "structurally fragile" line, not just the loss line.
- **It is an operations-actionable floor.** Below ~10% gross margin a route-day
  contributes almost nothing to fixed overhead (depot, truck financing, admin); at
  or above it the route is at least carrying its weight. That makes 10% a number
  the ops team can defend in a route-rationalization conversation.

In [5]:
THRESHOLD = 10.0  # gross_margin_pct floor, justified from P10 above

silver = silver.withColumn("underperforming_flag", F.col("gross_margin_pct") < THRESHOLD)

# Human-readable reason: lead with the margin fact, then append the dominant
# contributing condition(s) so Part 2 can read "why" off the row directly.
reason = F.concat(
    F.lit("margin "), F.round("gross_margin_pct", 1).cast("string"),
    F.lit(f"% < {THRESHOLD}%"),
)
reason = F.when(F.col("gross_profit") < 0, F.concat(reason, F.lit("; negative gross profit"))).otherwise(reason)
reason = F.when(F.col("disposal_cost_share") > 0.40, F.concat(reason, F.lit("; disposal-heavy (>40% of cost)"))).otherwise(reason)
reason = F.when(F.col("completion_rate") < 0.90, F.concat(reason, F.lit("; low completion (<90%)"))).otherwise(reason)
reason = F.when(F.col("fuel_cost_share") > 0.30, F.concat(reason, F.lit("; fuel-heavy (>30% of cost)"))).otherwise(reason)

silver = silver.withColumn(
    "underperformance_reason",
    F.when(F.col("underperforming_flag"), reason).otherwise(F.lit(None).cast("string")),
)

print("Example flagged route-days with reasons:")
silver.filter("underperforming_flag").select(
    "route_date_key", "gross_margin_pct", "underperformance_reason"
).show(6, truncate=False)

Example flagged route-days with reasons:


+--------------+----------------+---------------------------------------------------------------------------+
|route_date_key|gross_margin_pct|underperformance_reason                                                    |
+--------------+----------------+---------------------------------------------------------------------------+
|RDK-0000002   |-10.71          |margin -10.7% < 10.0%; negative gross profit; disposal-heavy (>40% of cost)|
|RDK-0000009   |1.97            |margin 2.0% < 10.0%; disposal-heavy (>40% of cost)                         |
|RDK-0000017   |8.94            |margin 8.9% < 10.0%; disposal-heavy (>40% of cost)                         |
|RDK-0000019   |-30.54          |margin -30.5% < 10.0%; negative gross profit; disposal-heavy (>40% of cost)|
|RDK-0000020   |-3.58           |margin -3.6% < 10.0%; negative gross profit; disposal-heavy (>40% of cost) |
|RDK-0000050   |-0.33           |margin -0.3% < 10.0%; negative gross profit; disposal-heavy (>40% of cost) |
+---------

### Step 4 — route-level **persistent underperformer** (a coarser, separate rule)

The row-level flag answers *"was this day bad?"*. It must **not** be reused to judge
a route, because one storm day, one truck breakdown, or one heavy one-off haul can
push a single day below 10% on an otherwise healthy route. Tagging that route the
same as a structurally unprofitable one would mis-direct the ops team toward noise.

So a **route** is a *persistent underperformer* only if it fails on a durable,
volume-weighted basis:

> `route_id` is persistent-underperforming when **EITHER**
> (a) its **revenue-weighted** margin across *all* its days < 10%  — i.e.
>     `sum(gross_profit) / sum(total_revenue) * 100 < 10`  (one cheap big day can't
>     hide behind many tiny good days, and vice-versa), **OR**
> (b) **≥ 20%** of its route-days are individually below the 10% threshold — a route
>     that's in the red one day in five is chronically fragile even if a few huge
>     days average it out.

Weighting by revenue (not a simple mean of daily percentages) is deliberate: a route
with one $24k day at 5% and twenty $2k days at 50% is *actually* unprofitable in
dollar terms, which a naive average of percentages would miss.

In [6]:
route_w = Window.partitionBy("route_id")

silver = (
    silver
    .withColumn("_rt_rev",    F.sum("total_revenue").over(route_w))
    .withColumn("_rt_profit", F.sum("gross_profit").over(route_w))
    .withColumn("_rt_days",   F.count(F.lit(1)).over(route_w))
    .withColumn("_rt_under_days", F.sum(F.col("underperforming_flag").cast("int")).over(route_w))
)

silver = (
    silver
    .withColumn("route_weighted_margin_pct",
                F.when(F.col("_rt_rev") == 0, None)
                 .otherwise(F.col("_rt_profit") / F.col("_rt_rev") * 100))
    .withColumn("route_pct_days_under",
                F.when(F.col("_rt_days") == 0, None)
                 .otherwise(F.col("_rt_under_days") / F.col("_rt_days")))
    .withColumn("route_persistent_underperformer",
                (F.col("route_weighted_margin_pct") < THRESHOLD) |
                (F.col("route_pct_days_under") >= 0.20))
    .drop("_rt_rev", "_rt_profit", "_rt_days", "_rt_under_days")
)

print("Persistent-underperformer routes (distinct):")
(silver.filter("route_persistent_underperformer")
   .select("route_id", "route_weighted_margin_pct", "route_pct_days_under")
   .distinct().orderBy("route_weighted_margin_pct").show(10, truncate=False))

Persistent-underperformer routes (distinct):


+--------+-------------------------+--------------------+
|route_id|route_weighted_margin_pct|route_pct_days_under|
+--------+-------------------------+--------------------+
|RTE-0114|-15.278176780353839      |0.875               |
|RTE-0035|-12.999868098378995      |0.8878504672897196  |
|RTE-0061|-4.549176014494565       |0.7821782178217822  |
|RTE-0059|0.8366043536788995       |0.6576576576576577  |
|RTE-0117|7.251499915999847        |0.5978260869565217  |
|RTE-0081|9.148674593922781        |0.5803571428571429  |
|RTE-0020|9.271962216673737        |0.48514851485148514 |
|RTE-0070|9.877721621375265        |0.5148514851485149  |
|RTE-0057|12.516923551913436       |0.49019607843137253 |
|RTE-0069|14.858874213786045       |0.4111111111111111  |
+--------+-------------------------+--------------------+
only showing top 10 rows


### Step 5 — real counts under the chosen rule

In [7]:
total_days = silver.count()
under_days = silver.filter("underperforming_flag").count()
total_routes = silver.select("route_id").distinct().count()
persistent_routes = silver.filter("route_persistent_underperformer").select("route_id").distinct().count()

print("=== UNDERPERFORMANCE COUNTS (chosen rule: margin < 10%) ===")
print(f"route-days total              : {total_days}")
print(f"underperforming route-days    : {under_days} ({round(100*under_days/total_days, 2)}%)")
print(f"routes total                  : {total_routes}")
print(f"persistent-underperformer rts : {persistent_routes} ({round(100*persistent_routes/total_routes, 2)}%)")

=== UNDERPERFORMANCE COUNTS (chosen rule: margin < 10%) ===
route-days total              : 12000
underperforming route-days    : 1216 (10.13%)
routes total                  : 120
persistent-underperformer rts : 21 (17.5%)


## Part C — Write Silver + OPTIMIZE / ZORDER

Partition by `year`, `region` — the two coarsest, lowest-cardinality slice keys the
BI dashboard filters on first. Then ZORDER by `(bu, area, route_id)` so the
finer operational-hierarchy filters co-locate within each partition's files.

Note: contrary to a common belief that `ZORDER` is Databricks-only, **open-source
Delta Lake (>= 2.0, and 4.0 here) supports `OPTIMIZE ... ZORDER BY` natively.** We
still wrap it in try/except as requested and print whichever branch actually runs,
so the notebook is honest about what the engine did.

In [8]:
(
    silver.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("year", "region")
    .save(SILVER)
)
print("Silver written. rows:", spark.read.format("delta").load(SILVER).count())

try:
    res = spark.sql(f"OPTIMIZE delta.`{SILVER}` ZORDER BY (bu, area, route_id)")
    m = res.first()["metrics"]
    print("BRANCH: ZORDER SUCCEEDED on this engine (OSS Delta supports it).")
    print(f"  files added   : {m['numFilesAdded']}")
    print(f"  files removed : {m['numFilesRemoved']}")
    print(f"  partitions opt: {m['partitionsOptimized']}")
    print(f"  zOrderStats   : {m['zOrderStats']}")
except Exception as e:
    print("BRANCH: ZORDER not supported on this engine -> no-op. Plain OPTIMIZE compaction only.")
    print("  reason:", str(e).splitlines()[0])
    try:
        spark.sql(f"OPTIMIZE delta.`{SILVER}`")
        print("  fallback OPTIMIZE (compaction) succeeded.")
    except Exception as e2:
        print("  even plain OPTIMIZE unavailable:", str(e2).splitlines()[0])

Silver written. rows: 12000


BRANCH: ZORDER SUCCEEDED on this engine (OSS Delta supports it).
  files added   : 18
  files removed : 54
  partitions opt: 18
  zOrderStats   : Row(strategyName='all', inputCubeFiles=Row(num=0, size=0), inputOtherFiles=Row(num=54, size=4238265), inputNumCubes=0, mergedFiles=Row(num=54, size=4238265), numOutputCubes=18, mergedNumCubes=None)


In [9]:
print("=== DESCRIBE HISTORY silver/route_day_profitability ===")
spark.sql(f"DESCRIBE HISTORY delta.`{SILVER}`").select(
    "version", "operation", "operationParameters"
).show(truncate=80)
spark.stop()
print("Spark stopped.")

=== DESCRIBE HISTORY silver/route_day_profitability ===


+-------+---------+--------------------------------------------------------------------------------+
|version|operation|                                                             operationParameters|
+-------+---------+--------------------------------------------------------------------------------+
|      1| OPTIMIZE|{predicate -> [], zOrderBy -> ["bu","area","route_id"], clusterBy -> [], auto...|
|      0|    WRITE|                           {mode -> Overwrite, partitionBy -> ["year","region"]}|
+-------+---------+--------------------------------------------------------------------------------+

Spark stopped.
